In [161]:
from bs4 import BeautifulSoup
import requests
import re
import networkx as nx
import spacy
import time
import math
import pickle
import json
from composer import *
from ipysigma import Sigma

In [163]:
G=pickle.load(open("graphe_centroids_pot_seuil=1.3.pkl", 'rb'))
#S = prepare_sigma_with_composer_ids(G)
import math

# Définir une colormap bleu-rouge pour les poids des arêtes
import matplotlib
cmap = matplotlib.cm.get_cmap('viridis')

# Normaliser les poids pour la colormap
weights = np.array([d['weight'] for _, _, d in G.edges(data=True)])
w_min, w_max = weights.min(), weights.max()

for u, v, d in G.edges(data=True):
    norm_weight = (d['weight'] - w_min) / (w_max - w_min) if w_max > w_min else 0.5
    rgba = cmap(norm_weight)
    # Convertir en couleur hexadécimale
    d['color_edge'] = matplotlib.colors.rgb2hex(rgba)

layout = {}
grid_positions = {}

angle_step = 2 * math.pi / max(1,len(G.nodes()))
radius = 1
for i,node in enumerate(G.nodes()):
    angle = i * angle_step
    layout[node] = {
        "x": radius * math.cos(angle),
        "y": radius * math.sin(angle)
    }

# Calcul d'un score de PageRank pour ajuster la taille des noeuds par exemple
#pagerank_scores = nx.pagerank(G)

# Visualisation avec ipysigma
sigma = Sigma(
    graph=G,
    raw_node_color="color",
    raw_edge_color="color_edge",
    layout=layout,
    node_label="nom complet",
    default_edge_type="curve",
    label_font="cursive",
    node_size=G.degree()#pagerank_scores,  
)

sigma

/var/folders/6_/wjb9ddn965xckzzthm1x7rnr0000gn/T/ipykernel_5029/3888903031.py:7: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = matplotlib.cm.get_cmap('viridis')


Sigma(nx.Graph with 70 nodes and 385 edges)

In [165]:
def prepare_sigma_with_composer_ids(original_graph):
    """Relabels a graph using Composer.id as node labels and returns a Sigma widget. The new sigma compatible graph is named graph and returned by the function"""
    #id_map = {node: int(node.id.replace('Q','')) for node in original_graph.nodes()}
    id_map={node: str(node)+" - "+str(node.id) for node in original_graph.nodes()}
    graph = nx.relabel_nodes(original_graph, id_map)

    # Clean all node/edge attributes to make sure they're JSON serializable because the composer object was not accepted by ipysigma
    for n, d in graph.nodes(data=True):
        for k, v in d.items():
            if not isinstance(v, (str, int, float, bool, type(None))):
                d[k] = str(v)

    for u, v, d in graph.edges(data=True):
        for k, val in d.items():
            if not isinstance(val, (str, int, float, bool, type(None))):
                d[k] = str(val)

    layout = {
        node: {
            "x": math.cos(i * 2 * math.pi / len(graph)),
            "y": math.sin(i * 2 * math.pi / len(graph)),
        }
        for i, node in enumerate(graph)
    }

    #pagerank_scores = nx.pagerank(graph)

    #for k, v in pagerank_scores.items():
    #    assert isinstance(v, (int, float)), f"Non-numeric pagerank for node {k}: {v}"
    #for u, v, d in g.edges(data=True):
    for u, v, d in graph.edges(data=True):
        if d.get("source") == "chroma":
            #graph[str(u)+" - "+str(u.id)][str(v)+" - "+str(v.id)]["color_edge"] = "#3498DB"  # Blue for chroma
            #graph[str(u)+" - "+str(u.id)][str(v)+" - "+str(v.id)]["source"]="chroma"
            graph[u][v]["color_edge"] = "#3498DB"  # Blue for chroma
            graph[u][v]["source"]="chroma"
        elif d.get("source") == "chroma and influ":
            graph[u][v]["color_edge"] ="#2ECC71"
            graph[u][v]["source"]="chroma and influ" #Green for influ and chroma (the edge was present in both graphs)
        else:
            graph[u][v]["color_edge"] = "#E74C3C"  # Red for only influ (or default)
            graph[u][v]["source"]="influ"

    sigma = Sigma(
        graph=graph,
        layout=layout,
        default_edge_type="curve",
        node_border_color_from="node",
        raw_edge_color="color_edge",
        label_font="cursive",
        node_size=graph.out_degree()#pagerank_scores
    )

    return graph,sigma

In [167]:
def add_chroma_edges(G_influ,G_chroma):
    id_dict={}
    for n in G_influ.nodes():
        for m in G_chroma.nodes():
            if m not in id_dict:
                id_dict[m]=find_id(m.split(";")[-1].split(" ")[0]+" "+m.split(";")[0].strip())
                if "Rameau" in m:#faute d'orthographe dans erlangen
                    id_dict[m]='Q1145'
                elif "Boccherini" in m:#faute d'orthographe dans erlangen
                    id_dict[m]="Q189766"
            if n.id==id_dict[m]:
                n.description='erlangen'
                for neigh in G_chroma.neighbors(m):
                    #if neigh.id in map(get_id,G_influ.nodes())
                    if neigh not in id_dict:
                        id_dict[neigh]=find_id(neigh.split(";")[-1].split(" ")[0]+" "+neigh.split(";")[0].strip())
                    if id_dict[neigh] is not None:
                        for node in G_influ.nodes():
                            if node.id == id_dict[neigh]:
                                #G_influ.add_edge(n,node)
                                attrs = G_chroma.get_edge_data(m, neigh, default={})
                                if (n,node) not in G_influ.edges():
                                    G_influ.add_edge(n, node, **attrs, source="chroma")
                                    #G_influ.add_edge(node,n)
                                    #attrs = G_chroma.get_edge_data(neigh, m, default={})
                                else:
                                    G_influ.add_edge(n, node, **attrs, source="chroma and influ")
                                if (node,n) not in G_influ.edges():
                                    G_influ.add_edge(node, n, **attrs, source="chroma")
                                else:
                                    G_influ.add_edge(node, n, **attrs, source="chroma and influ")
                                break
    return G_influ

In [169]:
"""def collapse_same_composer_nodes(G):#for some reason (probably when preparing the graph for a sigma display) there exists several different nodes corresponding to the same composer, it is necessary to collapse them into one unique node in order not to falsify result
    List_of_nodes_to_remove = []
    Already_treated = set()
    for node1 in G:
        id1 = node1.split("-")[-1]
        for node2 in G:
            if node2 not in Already_treated :
                id2 = node2.split("-")[-1]
                if id1==id2:
                    List_of_nodes_to_remove.append(node2)
                    for edge in ...
        Already_treated.add(id1)
    for n in List_of_nodes_to_remove:
        G.remove_node(n)
    return G"""

def collapse_same_composer_nodes(G):
    from collections import defaultdict

    # Group nodes by their ID suffix
    id_to_nodes = defaultdict(list)
    for node in G.nodes:
        composer_id = node.split("-")[-1].strip()
        id_to_nodes[composer_id].append(node)

    for composer_id, nodes in id_to_nodes.items():
        if len(nodes) <= 1:
            continue
        main_node = nodes[0]

        for duplicate_node in nodes[1:]:
            # Redirect outgoing edges
            for neighbor in G.successors(duplicate_node):
                if G.has_edge(main_node, neighbor):
                    # Merge edge sources
                    existing_source = G[main_node][neighbor].get("source", "")
                    new_source = G[duplicate_node][neighbor].get("source", "")
                    if existing_source != new_source:
                        combined_source = " and ".join(sorted(set(existing_source.split(" and ") + new_source.split(" and "))))
                        G[main_node][neighbor]["source"] = combined_source
                else:
                    # Add edge and copy attributes
                    G.add_edge(main_node, neighbor, **G[duplicate_node][neighbor])

            # Redirect incoming edges
            for predecessor in G.predecessors(duplicate_node):
                if G.has_edge(predecessor, main_node):
                    existing_source = G[predecessor][main_node].get("source", "")
                    new_source = G[predecessor][duplicate_node].get("source", "")
                    if existing_source != new_source:
                        combined_source = " and ".join(sorted(set(existing_source.split(" and ") + new_source.split(" and "))))
                        G[predecessor][main_node]["source"] = combined_source
                else:
                    G.add_edge(predecessor, main_node, **G[predecessor][duplicate_node])

            # Remove the duplicate node
            G.remove_node(duplicate_node)

    return G


In [171]:
G=pickle.load(open("graphe_centroids_pot_seuil=1.3.pkl", 'rb'))
g=pickle.load(open("Q174606.pickle", 'rb'))
graph = add_chroma_edges(g,G)
pickle.dump(graph, open("merged_compo_graph.pickle", 'wb'))
graph,S=prepare_sigma_with_composer_ids(graph)
graph=collapse_same_composer_nodes(graph)

layout = {
        node: {
            "x": math.cos(i * 2 * math.pi / len(graph)),
            "y": math.sin(i * 2 * math.pi / len(graph)),
        }
        for i, node in enumerate(graph)
    }
for u, v, d in graph.edges(data=True):
    if d.get("source") == "chroma":
        #graph[str(u)+" - "+str(u.id)][str(v)+" - "+str(v.id)]["color_edge"] = "#3498DB"  # Blue for chroma
        #graph[str(u)+" - "+str(u.id)][str(v)+" - "+str(v.id)]["source"]="chroma"
        graph[u][v]["color_edge"] = "#3498DB"  # Blue for chroma
        graph[u][v]["source"]="chroma"
    elif d.get("source") == "chroma and influ":
        graph[u][v]["color_edge"] ="#2ECC71"
        graph[u][v]["source"]="chroma and influ" #Green for influ and chroma (the edge was present in both graphs)
    else:
        graph[u][v]["color_edge"] = "#E74C3C"  # Red for only influ (or default)
        graph[u][v]["source"]="influ"
sigma = Sigma(
        graph=graph,
        layout=layout,
        default_edge_type="curve",
        node_border_color_from="node",
        raw_edge_color="color_edge",
        label_font="cursive",
        node_size=graph.out_degree()#pagerank_scores
    )

pickle.dump(graph, open("merged_sigma_graph.pickle", 'wb'))
sigma

Sigma(nx.DiGraph with 1,596 nodes and 5,125 edges)

In [150]:
set_of_erlangen_composers={"Tomaso Albinoni","Arcangelo Corelli ","George Frederic Handel", "Jean-Baptiste Lully","Henry Purcell","Antonio Vivaldi","Johann Christian Bach","Luigi Boccherini","Johann Michael Haydn","Joseph Haydn","Ignace Joseph Pleyel","Antonio Salieri","George Antheil","Alban Berg","Benjamin Britten","Paul Hindemith","Charles Edward Ives","Olivier Messiaen","Edgar Varese","Anton Webern","Kurt Weill","Hector Berlioz","Alexander Borodin","Anton Bruckner","Antonin Dvorak","Edvard Grieg","Franz Liszt","Modest Mussorgsky","Nicolai Rimsky-Korsakov","Camille Saint-Saens","Bedrich Smetana","Giuseppe Verdi","Richard Wagner","Johann Sebastian Bach","Francois Couperin","Lodovico Giustini","Giovanni Benedetto Platti","Jean-Philippe Rameau","Domenico Cimarosa","Muzio Clementi","Jan Ladislav Dussek","Joseph Haydn","Wolfgang Amadeus Mozart","Anton Webern","Bela Bartok","Pierre Boulez","Paul Hindemith","Olivier Messiaen","Darius Milhaud","Sergej Prokofiew","Arnold Schoenberg","Dmitri Shostakovich","Igor Stravinsky","Johannes Brahms","Frederic Chopin","Clara Schumann","Gabriel Fauré","Edvard Grieg","Franz Liszt","Felix Mendelssohn-Bartholdy","Robert Schumann","Peter Iljitsch Tchaikovsky","Carl Philipp Emanuel Bach","Gustav Mahler","Leopold Mozart","Gioacchino Rossini","Johann Stamitz","Richard Strauss","Georg Philipp Telemann","Ludwig van Beethoven","Carl Philipp Emanuel Bach","Claude Debussy","Maurice Ravel","Domenico Scarlatti","Franz Schubert", "Jean Sibelius","Carl Maria von Weber"}
def filter_graph_by_composer_list(graph, composers):
    # Find matching nodes in the graph
    composer_ids=list(map(find_id,composers))
    print(composer_ids)
    print(len(set_of_erlangen_composers))
    print(len(composer_ids))
    nodes_to_keep = [
        node for node in graph.nodes()
        if any(node.endswith(cid) for cid in composer_ids)
    ]

    subgraph = graph.subgraph(nodes_to_keep).copy()

    # Optionally: remove edges not both in the filtered set
    subgraph.remove_edges_from([
        (u, v) for u, v in subgraph.edges()
        if u not in nodes_to_keep or v not in nodes_to_keep
    ])

    return subgraph

filtered=filter_graph_by_composer_list(graph,set_of_erlangen_composers)

sg = Sigma(
    graph=filtered,
    layout=layout,  # or use your custom layout dict
    raw_node_color="color_node",
    raw_edge_color="color_edge",
    default_edge_type="curve",
    node_label="label",
    node_border_color_from="node"
)

sg


['Q174606', 'Q309648', 'Q45682', 'Q154770', 'Q51088', 'Q1192', 'Q41309', 'Q193673', 'Q156193', 'Q13894', 'Q1340', 'Q164004', 'Q4700', 'Q49481', 'Q189766', 'Q1151', 'Q93227', 'Q1511', 'Q7314', 'Q1339', 'Q132682', 'Q150767', 'Q186264', 'Q1178', 'Q711089', 'Q80135', 'Q55004', 'Q1268', 'Q7294', 'Q254', 'Q80621', 'Q150445', 'Q9726', 'Q1145', 'Q314019', 'Q106641', 'Q191775', 'Q57244', 'Q7304', 'Q156033', 'Q7349', 'Q48173', 'Q57247', 'Q104919', 'Q490381', 'Q7351', 'Q357172', 'Q164475', 'Q50186', 'Q156280', 'Q9695', 'Q83326', 'Q81752', 'Q262427', 'Q78475', 'Q7312', 'Q46096', 'Q167837', 'Q7317', 'Q151593', 'Q7315', 'Q255', 'Q137739', 'Q132232', 'Q7298', 'Q76428', 'Q154812', 'Q127030740', 'Q190933', 'Q75854']
70
70


Sigma(nx.DiGraph with 69 nodes and 1,060 edges)